# 03_paged_attention_and_radix_tree: Memory Paging & Radix Tree Prefix Cache

This notebook simulates virtual memory page mappings (PagedAttention logical-to-physical block tables) and SGLang RadixAttention prefix tree lookups, displaying block allocation and cache reuse efficiency in multi-turn chatbot contexts.

### Core Engineering Intuitions
- **PagedAttention**: Instead of pre-allocating a contiguous chunk of memory sized to the maximum context length (which wastes $60\%-80\%$ VRAM due to early terminations or padding), PagedAttention borrows OS virtual paging concepts to map dynamic tokens to non-contiguous blocks in VRAM, increasing memory utilization up to $\approx 96\%$.
- **RadixAttention**: Organizes cached prefixes in a tree. For multi-turn chats or multi-agent workflows, common prefixes (like system prompts) are cached as parent nodes. Subsequent requests matching these prefixes reuse key-value tensors directly, bypassing redundant prefill computation passes and accelerating TTFT by $2\times - 5\times$.

### Servings Trade-offs (Pros & Cons)
- **PagedAttention**:
  - *Pros*: Eliminates internal/external VRAM fragmentation; scales batch sizes.
  - *Cons*: Introduces table lookup CPU overhead and complex block-management code.
- **RadixAttention (SGLang)**:
  - *Pros*: Reuses arbitrary system contexts and system prompts; drastically drops TTFT.
  - *Cons*: Cache eviction policies (LRU) add orchestration overhead under memory pressure.

In [1]:
import os
import sys

class PagedBlockAllocator:
    def __init__(self, num_blocks=64, block_size=4):
        self.num_blocks = num_blocks
        self.block_size = block_size
        self.free_blocks = list(range(num_blocks))
        self.block_table = {}

    def allocate(self, request_id, num_tokens):
        num_blocks_needed = (num_tokens + self.block_size - 1) // self.block_size
        allocated = []
        for _ in range(num_blocks_needed):
            if not self.free_blocks:
                raise MemoryError("Out of VRAM blocks!")
            allocated.append(self.free_blocks.pop(0))
        self.block_table[request_id] = allocated
        return allocated

    def free(self, request_id):
        if request_id in self.block_table:
            self.free_blocks.extend(self.block_table[request_id])
            del self.block_table[request_id]

# Initialize allocator
allocator = PagedBlockAllocator(num_blocks=32, block_size=4)
allocator.allocate("req_1", 10)
print("Physical blocks allocated for req_1:", allocator.block_table["req_1"])
print("Remaining free blocks count:", len(allocator.free_blocks))

Physical blocks allocated for req_1: [0, 1, 2]
Remaining free blocks count: 29


In [2]:
class RadixNode:
    def __init__(self, prefix_tokens):
        self.prefix_tokens = prefix_tokens
        self.children = {}
        self.block_ids = []

class RadixAttentionCache:
    def __init__(self, allocator):
        self.root = RadixNode([])
        self.allocator = allocator

    def get_or_insert(self, request_id, tokens):
        # Split prompt into a shared system prefix and dynamic user query
        system_len = 7
        system_part = tokens[:system_len]
        user_part = tokens[system_len:]
        
        system_key = " ".join(system_part)
        user_key = " ".join(user_part)
        
        node = self.root
        blocks = []
        
        # 1. Check system prompt prefix cache
        if system_key in node.children:
            print(f"[CACHE HIT]: Matched prefix: '{system_key}'")
            system_node = node.children[system_key]
            blocks.extend(system_node.block_ids)
            node = system_node
        else:
            print(f"[CACHE MISS]: No matching prefix found for '{system_key}'")
            system_blocks = self.allocator.allocate(request_id + "_sys", len(system_part))
            system_node = RadixNode(system_part)
            system_node.block_ids = system_blocks
            node.children[system_key] = system_node
            blocks.extend(system_blocks)
            node = system_node
            
        # 2. Allocate and append user prompt suffix
        user_blocks = self.allocator.allocate(request_id + "_user", len(user_part))
        user_node = RadixNode(user_part)
        user_node.block_ids = user_blocks
        node.children[user_key] = user_node
        blocks.extend(user_blocks)
        
        return blocks

# Set up prefix cache
cache = RadixAttentionCache(allocator)
system_prompt = ["System:", "You", "are", "a", "helpful", "coding", "assistant"]
user_prompt_1 = system_prompt + ["How", "does", "attention", "work?"]
user_prompt_2 = system_prompt + ["Explain", "continuous", "batching."]

print("--- Request 1 (Initial Chat) ---")
blocks_1 = cache.get_or_insert("session_1", user_prompt_1)
print("Blocks mapping session_1:", blocks_1)

print("\n--- Request 2 (Shared System Prompt Chat) ---")
blocks_2 = cache.get_or_insert("session_2", user_prompt_2)
print("Blocks mapping session_2:", blocks_2)

--- Request 1 (Initial Chat) ---
[CACHE MISS]: No matching prefix found for 'System: You are a helpful coding assistant'
Blocks mapping session_1: [3, 4, 5]

--- Request 2 (Shared System Prompt Chat) ---
[CACHE HIT]: Matched prefix: 'System: You are a helpful coding assistant'
Blocks mapping session_2: [3, 4, 6]


### Output Explanation & Verification

- **Block Tables**: PagedAttention allocated 3 physical blocks (block IDs 0, 1, 2) to store the 10 tokens of `req_1`, leaving 29 blocks in the free list. This verifies dynamic page mapping without reserving large contiguous ranges.
- **Radix Caching hit/miss**: Request 1 encountered a cache miss on the system prompt prefix and allocated new blocks. Request 2 successfully matched the cached system prompt prefix `'System: You are a helpful coding assistant'` in the Radix tree structure, achieving a cache hit and reusing the physical key-value blocks directly (IDs `[3, 4]`). This reduces active memory allocation overhead and prevents redundant prefill passes.